# 属性 probe — invariant 化で属性は表現から消えたか（3 seed）

[classification_performance.ipynb](classification_performance.ipynb) は「性能は下がるのに gap は動かない」で
終わっている。**adversary がそもそも属性を消せているのか**が分かっていないので、checkpoint を後から probe する。
学習ログには属性予測損失が残っていない（`metrics/fit.json` が持つのは `train/loss` の合計だけ）。

測るのは分類 head の直前の表現（2048 次元）。backbone は凍結し、ここで学習するのは probe の重みだけになる。

**手順は post-hoc attacker の慣行に合わせる**（Elazar & Goldberg 2018 ほか）。probe は
**train で学習し、val で止めどきを決め、test で報告する**。adversary が消したと主張したい側の分析なので、
**probe は強いほうへ倒す**。probe が弱いせいで下がった分を「属性が消えた」と読まないため。

probe は 2 つ。**linear** は表現の線形分離性、**mlp**（hidden 256）は学習時の adversary と同じ容量で
「消し切れたか」を問い直す。層の深さ以外は同じ手順で学習する。

**adversary の対象かどうかで読み分ける。**

| 属性 | adversary | 読み方 |
| --- | --- | --- |
| sex / race / age / age_group_65 | 対象 | 下がっていれば adversary が効いている |
| ethnicity | 対象外 | ここまで下がるなら、属性固有ではなく表現全体が痩せている |
| frontal_lateral | 対象外 | 画像から直に読める撮影方向。probe 自体が機能していることの確認 |

**この notebook が言わないこと。** probe が下がっても下流の公平性が改善するとは限らない
（[reports/overall_comparison.md](reports/overall_comparison.md) の 2 節がその実例）。
control task / selectivity（Hewitt & Liang 2019）は取っていないので、同じ probe を両手法に当てた
**手法間の差**は読めるが、絶対値は読めない。race は White / Asian / Black の 3 群に絞ってあり、
学習時の adversary が見ていた 6 カテゴリより易しい問題になっている。**adversary の損失と並べない。**

```bash
uv run python analysis/common/predictions.py --study initial_resnet_vs_invariant --split train --features \
  --run-dir projects/hypernet_e2e/runs/20260922T063725Z-resnet-chexpert-s43-efcd  # 他 5 run と val / test も同様
uv run python analysis/initial_resnet_vs_invariant/attribute_probe.py
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.style
import numpy as np
import pandas as pd
import rootutils

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)

from analysis.common.paths import STYLE_SHEET  # noqa: E402

matplotlib.style.use(STYLE_SHEET)

RESULTS = ROOT / "analysis" / "initial_resnet_vs_invariant" / "results"
SPLIT = "test"
RESNET, INVARIANT = "ResNet", "attribute-invariant ResNet"
# 色は手法を表す。classification_performance.ipynb と同じ割り当てにして、2 つの notebook を並べて読めるようにする。
MODEL_COLOR = {RESNET: "#0173B2", INVARIANT: "#DE8F05"}
SEED_MARK = {"color": "#2b2b2b", "s": 13, "zorder": 4, "linewidths": 0}
CHANCE_LINE = {"color": "#52514e", "linestyle": (0, (4, 3)), "linewidth": 1.2, "zorder": 5}

probe = pd.read_csv(RESULTS / f"attribute_probe_{SPLIT}.csv")
probe_by_method = pd.read_csv(RESULTS / f"attribute_probe_by_method_{SPLIT}.csv")

CATEGORICAL = list(dict.fromkeys(probe[probe["num_classes"] > 0]["attribute"]))
ADVERSARIAL = [name for name in CATEGORICAL if probe[probe["attribute"] == name]["adversary"].iloc[0]]
CONTROL = [name for name in CATEGORICAL if name not in ADVERSARIAL]
SEEDS = sorted(probe["seed"].unique())
PROBES = list(dict.fromkeys(probe["probe"]))

## 描画と取り出しの共通部分

図は「属性を横に並べ、手法別の棒に平均±SD のひげと seed の点を重ね、chance を破線で置く」形しか無い。
描画を 1 つと、表から値を取り出す関数を 2 つ持つ（再利用も一括出力もしないので `plots.py` には出さない、
[`analysis/AGENTS.md`](../AGENTS.md)）。

In [ ]:
def probe_stats(summary, probe_kind, metric, attributes):
    """集約表から、属性順の平均・SD・chance を手法別に取り出す。"""
    rows = summary[(summary["probe"] == probe_kind) & (summary["metric"] == metric)]
    mean, std = {}, {}
    for model in MODEL_COLOR:
        indexed = rows[rows["model"] == model].set_index("attribute")
        mean[model] = indexed.loc[attributes, "mean"].tolist()
        std[model] = indexed.loc[attributes, "std"].tolist()
    chance = rows.set_index("attribute").loc[attributes, "chance"].groupby(level=0, sort=False).first()
    return mean, std, chance.tolist()


def probe_points(frame, probe_kind, metric, attributes):
    """per-seed の表から、属性ごとの seed 値を手法別に集める。"""
    rows = frame[(frame["probe"] == probe_kind) & (frame["metric"] == metric)]
    points = {}
    for model in MODEL_COLOR:
        part = rows[rows["model"] == model]
        points[model] = [part[part["attribute"] == name]["value"].tolist() for name in attributes]
    return points


def probe_bars(axis, categories, mean, std, points, chance, title):
    """属性ごとに手法別の棒（平均±SD）を並べ、seed の点と chance の破線を重ねる。"""
    positions = np.arange(len(categories))
    width = 0.8 / len(mean)
    offsets = np.linspace(-0.4 + width / 2, 0.4 - width / 2, len(mean))
    for offset, model in zip(offsets, mean, strict=True):
        centers = positions + offset
        error = {"ecolor": "#52514e", "elinewidth": 1.0, "capsize": 3}
        axis.bar(centers, mean[model], width, yerr=std[model], color=MODEL_COLOR[model], label=model, error_kw=error)
        for center, values in zip(centers, points[model], strict=True):
            axis.scatter(np.full(len(values), center), values, **SEED_MARK)
    for position, level in zip(positions, chance, strict=True):
        axis.plot([position - 0.42, position + 0.42], [level, level], **CHANCE_LINE)
    axis.set_xticks(positions, categories)
    axis.set_title(title)
    axis.margins(y=0.20)


def legend_on_top(figure, axis, title, note):
    """panel から凡例を取り出し、図の読み方を添えて figure の上に置く。"""
    handles, labels = axis.get_legend_handles_labels()
    figure.legend(handles, labels, loc="upper center", ncol=len(labels), bbox_to_anchor=(0.5, 1.10))
    figure.suptitle(f"{title}   ({note}, dots: each of {len(SEEDS)} seeds, dashes: chance)", y=1.18, fontsize=12)

## 属性は読めるか

まず全体を 1 枚の表で見る。**chance 列を必ず並べる。** probe の絶対値だけでは、下がったのが
「消えたから」なのか「元から読めていないから」なのかが区別できない。

`Δ mean` は invariant − ResNet。**負なら invariant 側で読みにくくなっている**。

In [ ]:
keys = ["adversary", "attribute", "probe", "metric"]
summary = probe_by_method.pivot(index=keys, columns="model", values=["mean", "std"])
summary.columns = [f"{model} {statistic}" for statistic, model in summary.columns]
summary["Δ mean"] = summary[f"{INVARIANT} mean"] - summary[f"{RESNET} mean"]
chance = probe_by_method.groupby(keys)["chance"].first()
support = probe.groupby("attribute")[["n_fit", "n_eval"]].first()
summary.join(chance).round(4).join(support, on="attribute").sort_index(ascending=[False, True, True, True])

per-seed の値も並べる。n=3 の平均は 1 本の外れで動くので、3 点がどう散っているかを先に見る。
ここは **mlp probe の balanced accuracy**（adversary と同じ容量で測った、消え残りの量）に絞る。

In [ ]:
rows = probe[(probe["probe"] == "mlp") & (probe["metric"] == "balanced accuracy")]
rows.pivot(index="seed", columns=["model", "attribute"], values="value").round(4)

### 図で見る

左が adversary の対象、右が対象外。**対象だけが下がっていれば adversary が効いている**と読める。
両方下がっていれば、属性固有ではなく表現全体が痩せている。
ひげ（SD）が重なっている属性では、平均の差を手法の差として読めない。

In [ ]:
METRIC = "balanced accuracy"
# panel title は figure の中なので英語にする。style sheet の font に日本語の字形が無く、混ぜると字が落ちる。
groups = [(ADVERSARIAL, "adversary targets"), (CONTROL, "controls")]
figure, axes = plt.subplots(
    len(PROBES), len(groups), figsize=(11.0, 7.2), sharey="row", width_ratios=[len(ADVERSARIAL), len(CONTROL)]
)
for row, probe_kind in zip(axes, PROBES, strict=True):
    for axis, (attributes, label) in zip(row, groups, strict=True):
        mean, std, chance = probe_stats(probe_by_method, probe_kind, METRIC, attributes)
        points = probe_points(probe, probe_kind, METRIC, attributes)
        probe_bars(axis, attributes, mean, std, points, chance, f"{probe_kind} probe — {label}")
    row[0].set_ylabel(METRIC)
legend_on_top(figure, axes[0][0], f"Attribute probe ({SPLIT} split)", "bar: mean ± SD")
plt.show()

### AUROC でも見る

balanced accuracy はある動作点での当たり方、AUROC は順序だけを見る。閾値の都合で片方だけ動くことがあるので、
両方が同じ向きに動いているかを確かめる。

In [ ]:
METRIC = "AUROC"
figure, axes = plt.subplots(
    len(PROBES), len(groups), figsize=(11.0, 7.2), sharey="row", width_ratios=[len(ADVERSARIAL), len(CONTROL)]
)
for row, probe_kind in zip(axes, PROBES, strict=True):
    for axis, (attributes, label) in zip(row, groups, strict=True):
        mean, std, chance = probe_stats(probe_by_method, probe_kind, METRIC, attributes)
        points = probe_points(probe, probe_kind, METRIC, attributes)
        probe_bars(axis, attributes, mean, std, points, chance, f"{probe_kind} probe — {label}")
    row[0].set_ylabel(METRIC)
legend_on_top(figure, axes[0][0], f"Attribute probe ({SPLIT} split)", "bar: mean ± SD")
plt.show()

## age は回帰のまま見る

adversary は age を MSE で消しにいくので、probe も回帰で測る。`age_group_65` は同じ age から作った
派生量で、他の属性と同じ物差し（balanced accuracy）で読むために上の表にも入れてある。

**MAE を併記する。** R2 は「元の分散のどれだけを説明できたか」しか言わず、**何歳ぶん当たるのか**が読めない。
chance は「fit split の中央値を常に答えた場合」の MAE。

In [ ]:
age = probe_by_method[probe_by_method["attribute"] == "age"]
table = age.pivot(index=["probe", "metric"], columns="model", values=["mean", "std"])
table.columns = [f"{model} {statistic}" for statistic, model in table.columns]
table["Δ mean"] = table[f"{INVARIANT} mean"] - table[f"{RESNET} mean"]
table.join(age.groupby(["probe", "metric"])["chance"].first()).round(4)

---

## 読み取り

まとめは [reports/attribute_probe.md](reports/attribute_probe.md) に置く。

**1. adversary は属性を消せていない。** mlp probe（学習時の adversary と同じ hidden 256）の
balanced accuracy は sex 0.886 → 0.871、age_group_65 0.756 → 0.741、race 0.557 → 0.551。
age は MAE 9.71 → 10.00 歳（chance 14.38 歳）。**chance からの上積みで見ると 94〜98% が残る。**
invariant 化した表現からも、sex は 87% で当たり、年齢は 10 歳の誤差で当たる。

**2. 下がり方は対照属性と大きくは違わない。** adversary の対象外である ethnicity も 0.619 → 0.616 と
下がる。sex と age_group_65 の下げ幅（−0.016 / −0.015）はその 4 倍ほどあり、3 seed の範囲も重ならないので
**adversary が「少しは」効いているとは言える**。ただし race と age は seed の範囲が重なり、符号も定まらない。

**3. probe が弱いせいではない。** frontal_lateral は両手法とも 0.999 で、probe 自体は機能している。
probe の学習行も 12.9 万〜17.8 万行あり、2048 次元に対して足りている。

**4. これで [reports/overall_comparison.md](reports/overall_comparison.md) の「性能は下がるのに gap が
動かない」が説明できる。** 払った性能（balanced accuracy −0.0038）は属性の除去に化けていない。
gap が動かないのは公平性指標の問題ではなく、**そもそも表現が変わっていない**ためになる。

他に効いている条件:

- **race は White / Asian / Black の 3 群に絞ってある。** 学習時の adversary は 6 カテゴリを見ている。
  同じ問題ではないので、**この表の値を adversary の損失と突き合わせない**
- **control task / selectivity（Hewitt & Liang 2019）は取っていない。** 同じ probe を両手法に当てた
  手法間の差は読めるが、絶対値（「87% は読めているうち」なのか）は読めない
- mlp probe は hidden 256 の 1 本だけ。容量を上げれば読める分は増えうるが、**ここでの結論は
  「消えていない」の向き**なので、容量を上げても向きは変わらない
- probe の初期値は 6 run で共通（`PROBE_SEED = 0`）。run 間の差に probe の引き当てを混ぜていない
- 6 本とも `trainer.deterministic=False`。同じ seed でも完全再現はしない
